# Bike Dataset Refit Test (Locked vs Unlocked Features)

This notebook tests the `IGANN_interactive.fit_from_shape_functions(...)` flow on the bike dataset.

It checks that:
1. A locked feature keeps its edited shape unchanged after refit.
2. An unlocked feature can change due to residual boosting.

In [ ]:
from copy import deepcopy
from pathlib import Path
import sys

import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

# Ensure local igann repo is importable (new shape-based refit path lives there)
igann_repo = Path.home() / "projects" / "igann"
if str(igann_repo) not in sys.path:
    sys.path.insert(0, str(igann_repo))

from igann import IGANN_interactive
from bike_preprocessing import preprocess_bike_hourly

In [ ]:
seed = 3
points = 120

X_proc, y_full, cat_info, labels = preprocess_bike_hourly(seed)
X_train_df, X_test_df, y_train_arr, y_test_arr = train_test_split(
    X_proc, y_full, test_size=0.2, random_state=seed
)
y_train = np.asarray(y_train_arr, dtype=float).reshape(-1)

print("Train shape:", X_train_df.shape)
print("Categorical:", sorted(cat_info.keys()))

In [ ]:
# 1) Train an initial interactive model to get baseline shapes
base_model = IGANN_interactive(
    task="regression",
    n_estimators=120,
    boost_rate=0.1,
    init_reg=1.0,
    elm_alpha=1.0,
    early_stopping=30,
    random_state=seed,
    verbose=0,
    scale_y=True,
    GAMwrapper=True,
    GAM_detail=points,
)
base_model.fit(X_train_df, y_train)
base_shapes = deepcopy(base_model.GAM.get_feature_dict())

# Pick one numeric feature as locked and one numeric feature as unlocked
numeric_features = [k for k, v in base_shapes.items() if v.get("datatype") == "numerical"]
if len(numeric_features) < 2:
    raise RuntimeError("Need at least two numerical features for this test.")

locked_feature = numeric_features[0]
unlocked_feature = numeric_features[1]

edited_feature_dict = deepcopy(base_shapes)

# Apply a visible edit to the locked feature (this should stay fixed after refit)
edited_feature_dict[locked_feature]["y"] = [float(y) + 0.75 for y in edited_feature_dict[locked_feature]["y"]]

# Apply a gentle edit to one unlocked feature (this may change after refit)
x_unlocked = np.asarray(edited_feature_dict[unlocked_feature]["x"], dtype=float)
y_unlocked = np.asarray(edited_feature_dict[unlocked_feature]["y"], dtype=float)
if x_unlocked.size > 0:
    span = max(float(x_unlocked.max() - x_unlocked.min()), 1e-9)
    wave = 0.3 * np.sin((x_unlocked - float(x_unlocked.min())) / span * 2 * np.pi)
    edited_feature_dict[unlocked_feature]["y"] = (y_unlocked + wave).tolist()

print("Locked feature:", locked_feature)
print("Unlocked feature:", unlocked_feature)

In [ ]:
# 2) Build NEW model from edited shapes, boost only unlocked features
refit_model = IGANN_interactive(
    task="regression",
    n_estimators=120,
    boost_rate=0.1,
    init_reg=1.0,
    elm_alpha=1.0,
    early_stopping=30,
    random_state=seed,
    verbose=0,
    scale_y=True,
    GAMwrapper=True,
    GAM_detail=points,
)

fit_cols = [c for c in X_train_df.columns if c != locked_feature]
refit_model.fit_from_shape_functions(
    X_train_df[fit_cols],
    y_train,
    edited_feature_dict,
)

combined_shapes = refit_model.GAM.get_feature_dict()

locked_before = np.asarray(edited_feature_dict[locked_feature]["y"], dtype=float)
locked_after = np.asarray(combined_shapes[locked_feature]["y"], dtype=float)
locked_diff = float(np.max(np.abs(locked_before - locked_after))) if len(locked_before) else 0.0

unlocked_before = np.asarray(edited_feature_dict[unlocked_feature]["y"], dtype=float)
unlocked_after = np.asarray(combined_shapes[unlocked_feature]["y"], dtype=float)
unlocked_diff = float(np.max(np.abs(unlocked_before - unlocked_after))) if len(unlocked_before) else 0.0

print("Refit feature columns:", refit_model._refit_feature_cols)
print("locked max |before-after|:", round(locked_diff, 8))
print("unlocked max |before-after|:", round(unlocked_diff, 8))

assert locked_diff < 1e-9, "Locked feature shape changed after refit"
assert unlocked_diff > 1e-6, "Unlocked feature did not change (increase class n_estimators if needed)"

print("Assertions passed.")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Locked feature: should overlap exactly
x_locked = np.asarray(combined_shapes[locked_feature]["x"], dtype=float)
axes[0].plot(x_locked, edited_feature_dict[locked_feature]["y"], label="edited", linewidth=2)
axes[0].plot(x_locked, combined_shapes[locked_feature]["y"], "--", label="after refit", linewidth=2)
axes[0].set_title(f"Locked: {locked_feature}")
axes[0].legend()

# Unlocked feature: expected to differ
x_unlocked_after = np.asarray(combined_shapes[unlocked_feature]["x"], dtype=float)
axes[1].plot(edited_feature_dict[unlocked_feature]["x"], edited_feature_dict[unlocked_feature]["y"], label="edited", linewidth=2)
axes[1].plot(x_unlocked_after, combined_shapes[unlocked_feature]["y"], "--", label="after refit", linewidth=2)
axes[1].set_title(f"Unlocked: {unlocked_feature}")
axes[1].legend()

plt.tight_layout()
plt.show()